# 07 — Path patching fiel ao artigo

O notebook 06 validou e diagnosticou a lista de heads publicada; este notebook testa conexões dirigidas entre essas classes já definidas. Demonstramos **Duplicate/Induction→S-Inhibition** via V em `S2` e **S-Inhibition→Name Movers** via Q em `END`, além de uma intervenção `multi_` com vários *senders* e *receivers*. O sender é sempre o write `result = z @ W_O` produzido por um head; o receiver é o componente Q, K ou V que lê essa contribuição.

Há dois métodos relacionados, mas não equivalentes. `paper_path_patch` reproduz o path patching histórico com **quatro passes**: (A) pABC para cachear o `result` novo do sender; (B) pIOI para cachear Q/K/V originais; (C) pIOI com o sender novo e os demais inputs congelados para capturar o receiver; (D) pIOI com somente esse receiver capturado. `freeze_mlps=False/True` decide se os MLPs do passe C são recomputados ou fixados no original, enquanto `allow_internal_interactions=False` congela os demais Q/K/V.

Mais adiante, `get_act_patch_direct_path` do TransformerLens injeta a diferença do write `z @ W_O` apenas no fork do receiver, mantém os outros leitores no write original e atua em todas as posições. Os resultados aparecem lado a lado como diagnósticos distintos; um método não é uma versão abreviada do outro.

In [1]:
import json
import os
from pathlib import Path
import subprocess

import pandas as pd
import plotly.express as px
import torch
from transformer_lens import patching as tl_patching

from ioi_replication.paper import (
    HeadPositionNode,
    PathReceiver,
    SemanticPosition,
    build_paired_dataset,
    logit_difference,
    paper_path_patch,
    published_path_grid_edges,
)
from ioi_replication.runtime import (
    compatibility_mode_config,
    load_gpt2_small,
    select_device,
)

SEED = 107
N_PATH = 64
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULT_PATH = PROJECT_ROOT / "results/paper/07_path_patching.json"

device = select_device()
model = load_gpt2_small(device)
# pIOI é a execução original; pABC fornece ativações novas com nomes substituídos.
dataset = build_paired_dataset(model.tokenizer, n=N_PATH, seed=SEED)
# Os tensores têm shape [N_PATH, tokens], e `positions` escolherá S2 ou END linha a linha nos hooks.
ioi_tokens = dataset.ioi_tokens.to(device)
abc_tokens = dataset.abc_tokens.to(device)

# Todos os métodos serão lidos na mesma diferença média bruta IO−S em END, sem normalização.
def metric(logits):
    return logit_difference(
        logits,
        dataset.io_token_ids.to(device),
        dataset.s_token_ids.to(device),
        dataset.positions["END"].to(device),
    )

# Esta baseline ancora `patched` e `effect` nas intervenções históricas e diretas abaixo.
print(f"device={device}; N={N_PATH}; baseline IO−S={metric(model(ioi_tokens)).item():.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

device=cuda; N=64; baseline IO−S=3.6031


## Intervenções históricas em posições semânticas

Cada sender (`result = z @ W_O`) e receiver abaixo traz sua posição explícita. Assim, `S2` e `END` são reunidos separadamente para cada linha do batch; nenhum hook substitui todas as posições.

Com `freeze_mlps=False`, os MLPs intermediários são recomputados e podem carregar parte do efeito do sender: esse é o confounder descrito no Appendix B. Com `freeze_mlps=True`, seus outputs no passe C são fixados nos valores originais, removendo essa rota. Em ambos os casos, `allow_internal_interactions=False` mantém os demais Q/K/V congelados.

Seguimos a convenção do notebook 06: `effect = patched − baseline`. Como inserimos o write de pABC no forward pIOI, efeito negativo significa perda de comportamento útil; ele não deve ser lido como o *recovery* positivo dos notebooks didáticos.

In [2]:
# Cada sender fornece o write `result = z @ W_O`; cada receiver seleciona um input Q/K/V de outro head.
interventions = {
    "duplicate_3.0_to_s_inhibition_7.3_v": {
        "senders": (HeadPositionNode(3, 0, SemanticPosition.S2),),
        "receivers": (PathReceiver(7, 3, "v", SemanticPosition.S2),),
    },
    "induction_5.5_to_s_inhibition_7.3_v": {
        "senders": (HeadPositionNode(5, 5, SemanticPosition.S2),),
        "receivers": (PathReceiver(7, 3, "v", SemanticPosition.S2),),
    },
    "s_inhibition_7.3_to_name_mover_9.9_q": {
        "senders": (HeadPositionNode(7, 3, SemanticPosition.END),),
        "receivers": (PathReceiver(9, 9, "q", SemanticPosition.END),),
    },
    "multi_duplicate_induction_to_s_inhibition_v": {
        "senders": (
            HeadPositionNode(0, 1, SemanticPosition.S2),
            HeadPositionNode(5, 5, SemanticPosition.S2),
        ),
        "receivers": (
            PathReceiver(7, 3, "v", SemanticPosition.S2),
            PathReceiver(7, 9, "v", SemanticPosition.S2),
        ),
    },
}

# Repetimos o algoritmo histórico de quatro passes com MLPs livres e com seus outputs congelados.
historical = {
    "freeze_mlps_false": {},
    "freeze_mlps_true": {},
}
for freeze_mlps, mode_name in (
    (False, "freeze_mlps_false"),
    (True, "freeze_mlps_true"),
):
    print(f"\n{mode_name}")
    for name, endpoints in interventions.items():
        senders = endpoints["senders"]
        receivers = endpoints["receivers"]
        # `freeze_mlps` só controla a rota pelos MLPs no passe C; os demais Q/K/V continuam congelados.
        result = paper_path_patch(
            model,
            ioi_tokens,
            abc_tokens,
            dataset.positions,
            dataset.positions,
            senders=senders,
            receivers=receivers,
            metric=metric,
            freeze_mlps=freeze_mlps,
            allow_internal_interactions=False,
        )
        # Guardamos endpoints e Δ IO−S para comparar depois com o direct path moderno.
        historical[mode_name][name] = {
            "baseline": result.baseline.item(),
            "patched": result.patched.item(),
            "effect": result.effect.item(),
            "senders": [
                {
                    "layer": sender.layer,
                    "head": sender.head,
                    "position": sender.position.value,
                }
                for sender in senders
            ],
            "receivers": [
                {
                    "layer": receiver.layer,
                    "head": receiver.head,
                    "position": receiver.position.value,
                    "component": receiver.component.value,
                }
                for receiver in receivers
            ],
        }
        print(
            f"{name}: {result.baseline.item():+.4f} → "
            f"{result.patched.item():+.4f} "
            f"(Δ={result.effect.item():+.4f})"
        )


freeze_mlps_false


duplicate_3.0_to_s_inhibition_7.3_v: +3.6031 → +3.5400 (Δ=-0.0630)


induction_5.5_to_s_inhibition_7.3_v: +3.6031 → +3.5434 (Δ=-0.0597)


s_inhibition_7.3_to_name_mover_9.9_q: +3.6031 → +3.5311 (Δ=-0.0719)


multi_duplicate_induction_to_s_inhibition_v: +3.6031 → +3.0647 (Δ=-0.5384)

freeze_mlps_true


duplicate_3.0_to_s_inhibition_7.3_v: +3.6031 → +3.5572 (Δ=-0.0458)


induction_5.5_to_s_inhibition_7.3_v: +3.6031 → +3.5528 (Δ=-0.0502)


s_inhibition_7.3_to_name_mover_9.9_q: +3.6031 → +3.5386 (Δ=-0.0644)


multi_duplicate_induction_to_s_inhibition_v: +3.6031 → +3.1508 (Δ=-0.4523)


## Diagnóstico moderno `get_act_patch_direct_path`

A comparação usa a mesma métrica bruta IO−S nos métodos e salva os endpoints separadamente. Não misturamos essa diferença de logits com uma *normalized metric*, que expressaria recuperação relativa entre baselines limpa e corrompida.

A semelhança termina no objetivo de isolar uma aresta. O path patching histórico produz o estado do receiver por uma sequência causal de quatro forwards e faz a troca apenas em `S2` ou `END` de cada exemplo; o modo `freeze_mlps` muda o que pode ser recomputado no terceiro passe. Já o helper do TransformerLens 4.3 não reproduz esses passes: ele soma a diferença do write `result = z @ W_O` somente no fork pré-Q/K/V do receiver, mantém **outros leitores** no write original e intervém em **todas as posições**.

As arestas singleton produzem efeitos pequenos, cerca de −0,05 a −0,07 IO−S, enquanto a intervenção `multi_` chega a aproximadamente −0,45/−0,54. Isso não é uma contradição: isolar uma aresta pergunta pelo efeito marginal daquele caminho; intervir em vários senders e receivers pergunta pelo efeito conjunto do grupo, incluindo redundância e compensação. O helper moderno aparece apenas nas três arestas singleton, pois sua API faz o sweep de um sender por vez.

In [3]:
# Primeiro, o método histórico percorre as 88 arestas publicadas e mantém sender/receiver semanticamente posicionados.
grid_rows = []
for edge in published_path_grid_edges():
    receiver = PathReceiver(
        edge.receiver.layer,
        edge.receiver.head,
        edge.receiver_component,
        edge.receiver_position,
    )
    result = paper_path_patch(
        model,
        ioi_tokens,
        abc_tokens,
        dataset.positions,
        dataset.positions,
        senders=(edge.sender,),
        receivers=(receiver,),
        metric=metric,
        freeze_mlps=False,
        allow_internal_interactions=False,
    )
    sender_label = (
        f"{edge.sender_class.value}:{edge.sender.layer}.{edge.sender.head}"
        f"@{edge.sender.position.value}"
    )
    receiver_label = (
        f"{edge.receiver_class.value}:{receiver.layer}.{receiver.head}"
        f".{receiver.component.value}@{receiver.position.value}"
    )
    grid_rows.append({
        "sender_class": edge.sender_class.value,
        "receiver_class": edge.receiver_class.value,
        "sender_label": sender_label,
        "receiver_label": receiver_label,
        "sender": {
            "layer": edge.sender.layer,
            "head": edge.sender.head,
            "position": edge.sender.position.value,
        },
        "receiver": {
            "layer": receiver.layer,
            "head": receiver.head,
            "component": receiver.component.value,
            "position": receiver.position.value,
        },
        "baseline": result.baseline.item(),
        "patched": result.patched.item(),
        "effect": result.effect.item(),
    })

assert len(grid_rows) == 88
sender_labels = list(dict.fromkeys(row["sender_label"] for row in grid_rows))
receiver_labels = list(dict.fromkeys(row["receiver_label"] for row in grid_rows))
effect_lookup = {
    (row["sender_label"], row["receiver_label"]): row["effect"]
    for row in grid_rows
}
effect_matrix = [
    [effect_lookup.get((sender, receiver)) for receiver in receiver_labels]
    for sender in sender_labels
]
grid_frame = pd.DataFrame(grid_rows)
grid_heatmap = grid_frame.pivot(
    index="sender_label",
    columns="receiver_label",
    values="effect",
).reindex(index=sender_labels, columns=receiver_labels)
fig_grid = px.imshow(
    grid_heatmap,
    labels={"x": "receiver", "y": "sender", "color": "Δ IO−S"},
    title="Grade sistemática das 88 arestas publicadas (quatro passes, MLPs livres)",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
)
fig_grid.update_layout(width=1200, height=760)
fig_grid.show()
published_path_grid = {
    "n_edges": len(grid_rows),
    "freeze_mlps": False,
    "allow_internal_interactions": False,
    "rows": grid_rows,
    "sender_labels": sender_labels,
    "receiver_labels": receiver_labels,
    "effect_matrix": effect_matrix,
}

# O helper lê z dos caches e aplica W_O para formar a diferença do sender `result = z @ W_O`.
sender_hook_names = sorted({
    f"blocks.{sender.layer}.attn.hook_z"
    for endpoints in interventions.values()
    for sender in endpoints["senders"]
})
_, abc_z_cache = model.run_with_cache(abc_tokens, names_filter=sender_hook_names)
_, ioi_z_cache = model.run_with_cache(ioi_tokens, names_filter=sender_hook_names)

# Direct path não executa os quatro passes: injeta esse write diferencial só no fork Q/K/V do receiver, em all-pos.
modern_direct = {}
for name, endpoints in interventions.items():
    senders = endpoints["senders"]
    receivers = endpoints["receivers"]
    if len(senders) != 1 or len(receivers) != 1:
        continue
    sender = senders[0]
    receiver = receivers[0]
    matrix = tl_patching.get_act_patch_direct_path(
        model=model,
        corrupted_tokens=ioi_tokens,
        clean_cache=abc_z_cache,
        corrupted_cache=ioi_z_cache,
        patching_metric=metric,
        src_layer=sender.layer,
        src_head=sender.head,
        component=receiver.component.value,
        verbose=False,
    )
    patched = matrix[receiver.layer, receiver.head]
    baseline_value = historical["freeze_mlps_false"][name]["baseline"]
    baseline = torch.tensor(baseline_value, device=patched.device)
    modern_direct[name] = {
        "baseline": baseline.item(),
        "patched": patched.item(),
        "effect": (patched - baseline).item(),
        "position_scope": "all_positions",
        "other_readers": "original source write",
    }
    print(
        f"{name}: {baseline.item():+.4f} → {patched.item():+.4f} "
        f"(Δ={patched.item() - baseline.item():+.4f}; all-pos)"
    )

# O gráfico reúne efeitos na mesma unidade bruta, mas os rótulos preservam freeze e escopo posicional distintos.
plot_rows = []
for name in interventions:
    plot_rows.extend([
        {
            "edge": name,
            "method": "paper semantic / MLPs recomputed",
            "effect": historical["freeze_mlps_false"][name]["effect"],
        },
        {
            "edge": name,
            "method": "paper semantic / MLPs frozen",
            "effect": historical["freeze_mlps_true"][name]["effect"],
        },
    ])
    if name in modern_direct:
        plot_rows.append({
            "edge": name,
            "method": "TL direct edge / all-pos",
            "effect": modern_direct[name]["effect"],
        })
fig = px.bar(
    plot_rows,
    x="edge",
    y="effect",
    color="method",
    barmode="group",
    title="Efeito bruto em IO−S: freeze de MLP e formulação direta",
    labels={"effect": "Δ IO−S", "edge": "aresta"},
)
fig.update_layout(width=1050, height=560, xaxis_tickangle=-20)
fig.show()

duplicate_3.0_to_s_inhibition_7.3_v: +3.6031 → +3.5573 (Δ=-0.0458; all-pos)


induction_5.5_to_s_inhibition_7.3_v: +3.6031 → +3.5526 (Δ=-0.0505; all-pos)


s_inhibition_7.3_to_name_mover_9.9_q: +3.6031 → +3.5380 (Δ=-0.0650; all-pos)


In [4]:
# A proveniência identifica as implementações exatas por commit e blob Git.
source_paths = {
    "patching": "src/ioi_replication/paper/patching.py",
    "circuit": "src/ioi_replication/paper/circuit.py",
    "runtime": "src/ioi_replication/runtime.py",
    "notebook": "notebooks/07_path_patching.ipynb",
}
source_commit = os.environ.get("IOI_SOURCE_COMMIT") or subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
source_blobs = {
    name: subprocess.check_output(
        ["git", "rev-parse", f"{source_commit}:{relative_path}"],
        cwd=PROJECT_ROOT,
        text=True,
    ).strip()
    for name, relative_path in source_paths.items()
}

# Os resultados históricos e modernos ficam em campos separados para não sugerir equivalência entre os métodos.
artifact = {
    "paper": "arXiv:2211.00593",
    "classification": "path-grid",
    "config": {
        "model_name": "gpt2",
        "seed": SEED,
        "n_examples": N_PATH,
        "dataset_sha256": dataset.dataset_sha256,
        "metric": "mean IO-minus-S logit difference at semantic END",
        "freeze_mlps_modes": [False, True],
        "allow_internal_interactions": False,
        "transformer_lens_compatibility_mode": compatibility_mode_config(),
    },
    "traceability": {
        "source_commit": source_commit,
        "hash_method": "git rev-parse <source_commit>:<path> (normalized Git blob IDs)",
        "source_blobs": source_blobs,
    },
    "historical_four_pass_semantic": historical,
    "published_path_grid": published_path_grid,
    "transformerlens_direct_path_all_positions": modern_direct,
    "comparison_warning": (
        "All sections use raw IO-S logit difference. The modern helper "
        "isolates a direct edge at all positions; historical four-pass "
        "results patch semantic positions and separately report MLP modes."
    ),
}
# A advertência serializada conserva a chave de leitura usada na comparação e encerra a sequência 05→07.
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULT_PATH.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
print(f"Resultado salvo em {RESULT_PATH.relative_to(PROJECT_ROOT)}")
artifact["config"]

Resultado salvo em results\paper\07_path_patching.json


{'model_name': 'gpt2',
 'seed': 107,
 'n_examples': 64,
 'dataset_sha256': '9fc55b67ce2b590bad8061f42ae93234627ad1430bf6649ea6165aa6fed64d91',
 'metric': 'mean IO-minus-S logit difference at semantic END',
 'freeze_mlps_modes': [False, True],
 'allow_internal_interactions': False,
 'transformer_lens_compatibility_mode': {'disable_warnings': False,
  'no_processing': False,
  'fold_ln': True,
  'center_writing_weights': True,
  'center_unembed': True,
  'fold_value_biases': True,
  'refactor_factored_attn_matrices': False}}

## Limites e próxima etapa

A grade sistemática usa N=64 e cobre as 88 combinações concretas das seis relações de classes publicadas. Ela diagnostica a topologia publicada; não faz descoberta cega nem reproduz numericamente todas as figuras do artigo.

A diferença entre os dois modos históricos quantifica quanto o resultado muda quando a rota por MLPs recomputados é permitida ou congelada. Ela não transforma um modo no helper moderno: o direct path mantém os outros leitores no original e isola uma aresta em todas as posições. Do mesmo modo, o efeito pequeno de uma aresta isolada e o efeito maior de uma intervenção múltipla respondem a perguntas diferentes sobre contribuição marginal e comportamento do grupo.

Essa diferença entre componentes individuais e efeito coletivo leva ao notebook 08: ele testa se Backup Name Movers compensam a remoção dos Name Movers primários e como essa redundância sustenta o comportamento IOI.